In [ ]:
# BDD FINALE FUSION

import pandas as pd
import os
import re

# Fichiers à intégrer
fichiers = {
    "BDD_Client_Borneo.csv": "Borneo",
    "BDD_Client_LuckyLikes.csv": "LuckyLikes",
    "BDD_Client_Dishop.csv": "Dishop",
    "clients_caisse_filtres.csv": "splash_caisse",
    "BDD_Client_SplashFidelite.csv": "splash-fidélité"
}

OUTPUT_FILE = "BDD_Clients_Finale.csv"
OUTPUT_FILE_EXPORT = os.getenv(
    "BDD_CLIENTS_FUSIONNEE_PATH",
    r"C:\path\to\BDD_Clients_Fusionnee.csv"
)

# Détection et normalisation des colonnes
def normaliser_dataframe(df):
    for col in df.columns:
        col_lower = col.lower()

        if col_lower in ["telephone", "téléphone", "numéro", "numero"]:
            df.rename(columns={col: "Téléphone"}, inplace=True)
        elif col_lower == "prenom":
            df.rename(columns={col: "Prénom"}, inplace=True)
        elif col_lower == "nom":
            df.rename(columns={col: "Nom"}, inplace=True)
        elif col_lower == "email":
            df.rename(columns={col: "Email"}, inplace=True)

    return df


# Nettoyage numéro de téléphone
def format_telephone(num):
    if pd.isna(num):
        return None

    num = re.sub(r"\D", "", str(num))

    if num.startswith("0") and len(num) == 10:
        return "+33" + num[1:]
    if num.startswith("33") and len(num) == 11:
        return "+" + num
    if num.startswith(("6", "7", "1", "2", "9")):
        return "+33" + num
    if num.startswith(("336", "337")):
        return "+" + num

    return None


# Fusion
df_final = pd.DataFrame()

for fichier, plateforme in fichiers.items():
    df = pd.read_csv(fichier)
    df = normaliser_dataframe(df)

    # Fusion prénom + nom si disponible
    if "Nom" in df.columns and "Prénom" in df.columns:
        df["Nom Complet"] = df["Prénom"].fillna("").astype(str) + " " + df["Nom"].fillna("").astype(str)
    elif "Prénom" in df.columns:
        df["Nom Complet"] = df["Prénom"]
    elif "Nom" in df.columns:
        df["Nom Complet"] = df["Nom"]
    else:
        df["Nom Complet"] = ""

    df["Nom Complet"] = df["Nom Complet"].str.strip()

    # Format téléphone
    if "Téléphone" in df.columns:
        df["Téléphone"] = df["Téléphone"].apply(format_telephone)
    else:
        df["Téléphone"] = None

    # Colonnes finales
    df["Plateforme"] = plateforme
    colonnes_finales = ["Nom Complet", "Email", "Téléphone", "Restaurant", "Plateforme"]

    for col in colonnes_finales:
        if col not in df.columns:
            df[col] = None

    df = df[colonnes_finales]

    # Ajout au dataset final
    df_final = pd.concat([df_final, df], ignore_index=True)

# Suppression des lignes sans numéro
df_final = df_final[df_final["Téléphone"].notna()]

# Suppression des doublons sur téléphone en gardant la ligne la plus complète
df_final["infos"] = df_final[["Nom Complet", "Email", "Téléphone", "Restaurant"]].notnull().sum(axis=1)
df_final = df_final.sort_values(by="infos", ascending=False)
df_final = df_final.drop_duplicates(subset="Téléphone", keep="first")
df_final = df_final.drop(columns="infos")

# Repasser au format français
df_final["Téléphone"] = df_final["Téléphone"].str.replace(r"^\+33", "0", regex=True)

# Sauvegardes
df_final.to_csv(OUTPUT_FILE, index=False, encoding="utf-8")
df_final.to_csv(OUTPUT_FILE_EXPORT, index=False, encoding="utf-8-sig", sep=";")

print("Fusion terminée.")
df_final.head(10)